# Node Classification with Additional Node Features

*August 4 2026*  
*Training Workshop: Introduction to Deep Graph Learning*  
*Ingo Scholtes, CAIDAS, Julius-Maximilians-Universität Würzburg (JMU), Germany*  

In the previous notebooks, we have considered networks where nodes did not have additional features, i.e. we used a one-hot encoding of the nodes as input to our GCN. With this approach, the GCN can only learn based on patterns contained in the graph topology. But what if we have additional features that could help the GCN, e.g., to classify nodes? Here we demonstrate this important capability of graph convolutional networks.

In [1]:
import numpy as np
import seaborn as sns
import torch
import torch_geometric
from matplotlib import pyplot as plt
from sklearn.decomposition import TruncatedSVD

import pathpyG as pp

plt.style.use('default')
sns.set_style("whitegrid")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Running on', device)

Running on cuda


We first use `pathpyG` to generate the synthetic network shown on slide 29.

In [ ]:
n = 50

random_1 = pp.algorithms.generative_models.watts_strogatz(n=n, s=4, p=0.01, mapping=pp.IndexMap([str(x) for x in range(n)]), undirected=True)
random_2 = pp.algorithms.generative_models.watts_strogatz(n=n, s=4, p=0.01, mapping=pp.IndexMap([str(x+1*n) for x in range(n)]), undirected=True)

network = random_1 + random_2
network = network.to_undirected()

# node features, cluster labels, and targets are assigned after merging and undirecting the
# graph, since to_undirected() does not preserve custom node attributes set beforehand
network.data.x = torch.cat((torch.tensor([0.]*int(n/2)), torch.tensor([1.]*int(n/2)),
                             torch.tensor([0.]*int(n/2)), torch.tensor([1.]*int(n/2))))
network.data.cluster = torch.cat((torch.tensor([0]*int(n/2)), torch.tensor([1]*int(n/2)),
                                   torch.tensor([2]*int(n/2)), torch.tensor([3]*int(n/2))))
network.data.y = torch.cat((torch.tensor([[0,0]]*int(n/2)), torch.tensor([[0,1]]*int(n/2)),
                             torch.tensor([[1,0]]*int(n/2)), torch.tensor([[1,1]]*int(n/2)))).float()

# randomly update src and tgt in network.edge_index so that link source in random_1 to target in random_2 and vice-versa
src_indices = torch.where(network.data.cluster < 2)[0]
tgt_indices = torch.where(network.data.cluster >= 2)[0]
num_edges_to_update = 10
edge_indices_to_update = np.random.choice(network.data.edge_index.shape[1], num_edges_to_update, replace=False)
for edge_index in edge_indices_to_update:
    src, tgt = network.data.edge_index[:, edge_index]
    if src in src_indices and tgt in src_indices:
        new_tgt = np.random.choice(tgt_indices.cpu().numpy())
        network.data.edge_index[1, edge_index] = new_tgt
    elif src in tgt_indices and tgt in tgt_indices:
        new_src = np.random.choice(src_indices.cpu().numpy())
        network.data.edge_index[0, edge_index] = new_src

# add one hot encoding + node features
from torch.nn.functional import one_hot as ohe

network.data.x = torch.cat((network.data.x.unsqueeze(1), ohe(torch.arange(0,2*n))), dim=1)

pp.plot(network, edge_color='gray');

We reuse the implementation of the GCN from the previous notebook:

In [3]:
class GraphConvolution(torch_geometric.nn.MessagePassing):

    def __init__(self, in_ch, out_ch):
        super().__init__(aggr='add')

        # this linear function is used to transform node features 
        # into messages that are then "sent" to neighbors
        self.linear = torch.nn.Linear(in_ch, out_ch)
        
    def forward(self, x, edge_index):
        """This function uses the edges captured in edge_index, performs
        the graph convolution function according to (Kipf, Welling 2017)
        and propagates the transformed features along the edges of the graph
        """
        # by adding self-loops, we ensure that aggregated messages from neighbors 
        # are combined with information from the node itself
        # this corresponds to matrix $\tilde{A}$ in (Kipf, Welling 2017)
        edge_index, _ = torch_geometric.utils.add_self_loops(edge_index, num_nodes=x.size(0))

        # we linearly transform the features of *all* nodes stored in x
        x = self.linear(x)

        # extract the source and target nodes of all edges
        source, target = edge_index
        
        # compute the (in-)degrees $d_i$ of source nodes
        deg = torch_geometric.utils.degree(target, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0

        # with this, the normalization to be applied in the propagation step can be expressed as
        # this corresponds to D^{-0.5} A * D^{-0.5} in (Kipf, Welling 2017)
        norm = deg_inv_sqrt[source] * deg_inv_sqrt[target]
        
        # the propagate function propagates messages along the edges of the graph
        # this function internally calls the functions: message(), aggregate() and update()
        # the normalization is applied in the message() function
        return self.propagate(edge_index, x=x, norm=norm)
    
    def message(self, x_j, norm):
        # x_j is a so-called **lifted** tensor which contains the source node features of each edge, 
        # i.e. it has a shape (m, out_ch) where m is the number of edges

        # a call to view(-1, 1) returns a reshaped tensor, where the second dimension 
        # is one and the first dimension is inferred automatically
        return norm.view(-1,1) * x_j

In [4]:
class GCN(torch.nn.Module):

    def __init__(self, data: torch_geometric.data.Data, out_ch, hidden_dim=16):
        super().__init__()

        self.input_to_hidden = GraphConvolution(data.num_node_features, hidden_dim)
        self.hidden_to_output =  GraphConvolution(hidden_dim, out_ch)
        
    def forward(self, x, edge_index):
        
        # first graph convolution -> map nodes to representations in hidden_dim dimensions
        x = self.input_to_hidden(x, edge_index)

        # non-linear activation function
        x = torch.sigmoid(x)

        # graph convolution -> maps node representations to output classes
        x = self.hidden_to_output(x, edge_index)

        return torch.sigmoid(x)

In [5]:
transform = torch_geometric.transforms.RandomNodeSplit(split='train_rest', num_val=0.3, num_test=0)
data = transform(network.data)

print(data)

Data(edge_index=[2, 800], num_nodes=100, node_sequence=[100, 1], x=[100, 101], cluster=[100], y=[100, 2], train_mask=[100], val_mask=[100], test_mask=[100])


In [6]:
model = GCN(data, out_ch=2, hidden_dim=4)

epochs = 200
lrn_rate = 0.1

optimizer = torch.optim.SGD(model.parameters(), lr=lrn_rate)

In [ ]:
indices = np.arange(100)
    
losses = []

model.train()
for epoch in range(epochs):

    error = 0
    print(epoch)
    
    np.random.shuffle(indices)
    for i in indices:

        if data.train_mask[i]:

            # set gradients to zero
            optimizer.zero_grad()

            # compute loss function for training sample and backpropagate
            output = model(network.data.x, network.data.edge_index)
            loss = torch.nn.functional.binary_cross_entropy(output[i], data.y[i])
            loss.backward()

            # update parameters
            optimizer.step()

            error += loss.detach().numpy()

    losses.append(error)

# plot evolution of loss function
plt.plot(range(epochs), losses);

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38


In [ ]:
output = model.forward(network.data.x, network.data.edge_index)

# we efficiently map probabilities to classes by rounding values to the 
# nearest integer, i.e. we obtain class 0 for probabilities smaller than 0.5
# and class 1 for probabilities larger than 0.5
prediction = output.round().long()

print(prediction)

In [ ]:
colors = {}
for v in network.nodes:
    index = network.mapping.to_idx(v)
    if data.val_mask[index]:
        if torch.equal(prediction[index], torch.tensor([0,0])):
            colors[v] = 'blue'
        elif torch.equal(prediction[index], torch.tensor([0, 1])):
            colors[v] = 'orange'
        elif torch.equal(prediction[index], torch.tensor([1, 0])):
            colors[v] = 'magenta'
        else:
            colors[v] = 'cyan'
    else:
     colors[v] = 'gray'
pp.plot(network, edge_color='gray', node_color=colors)

In [ ]:
embedding = model.input_to_hidden.forward(network.data.x, network.data.edge_index)

svd = TruncatedSVD()
low_dim = svd.fit_transform(embedding.detach().numpy())

colors = {}
for v in network.nodes:
    index = network.mapping.to_idx(v)
    if data.cluster[index] == 0:
        colors[index] = 'blue'
    elif data.cluster[index] == 1:
        colors[index] = 'orange'
    elif data.cluster[index] == 2:
        colors[index] = 'magenta'
    else:
        colors[index] = 'cyan'
plt.clf()
plt.scatter(low_dim[:,0], low_dim[:,1], c=colors.values());